In [1]:
import os
import tarfile
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets.utils import download_url
from transformers import (
    CLIPProcessor,
    CLIPModel,
    CLIPTokenizer,
    GPT2Tokenizer,
    GPT2Config,
    GPT2LMHeadModel
)
from tqdm import tqdm

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

Using Device: cuda


In [3]:
# def prepare_dataset():
#     if os.path.exists("Images"):
#         print("Dataset already exists. Skipping download.")
#         return

#     print("Downloading Stanford Dogs Dataset (750MB)...")
#     url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
#     download_url(url, root=".", filename="images.tar")

#     print("Extracting images...")
#     with tarfile.open("images.tar") as tar:
#         tar.extractall(path=".")

#     print("Generating VQA Pairs...")
#     data = []
#     base_dir = "Images"

#     for breed_folder in os.listdir(base_dir):
#         folder_path = os.path.join(base_dir, breed_folder)
#         if os.path.isdir(folder_path):
#             breed_name = breed_folder.split("-")[-1].replace("_", " ").lower()

#             for img_file in os.listdir(folder_path):
#                 rel_path = os.path.join(breed_folder, img_file)
#                 data.append({
#                     'image_name': rel_path,
#                     'question': "What breed of dog is this?",
#                     'answer': breed_name
#                 })

#     df = pd.DataFrame(data)
#     df = df.sample(frac=1, random_state=42).reset_index(drop=True)
#     df.to_csv("dog_vqa_dataset.csv", index=False)
#     print(f"Created Dataset with {len(df)} images.")

# prepare_dataset()

In [3]:
class ManualCrossAttention(nn.Module):
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.ln = nn.LayerNorm(dim)

    def forward(self, text_embeds, image_embeds):
        B, T_q, C = text_embeds.shape
        B, T_kv, _ = image_embeds.shape

        q = self.q_proj(text_embeds).view(B, T_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(image_embeds).view(B, T_kv, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(image_embeds).view(B, T_kv, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) * self.scale
        attn_probs = attn_scores.softmax(dim=-1)

        context = (attn_probs @ v).transpose(1, 2).contiguous()
        context = context.view(B, T_q, C)
        output = self.out_proj(context)

        return self.ln(text_embeds + output)

In [4]:
def crossGPT2(model_name = 'gpt2'):
    gpt2_config = GPT2Config.from_pretrained(model_name)
    gpt2_config.add_cross_attention = True
    gpt2_model = GPT2LMHeadModel.from_pretrained(model_name, config=gpt2_config)
    return gpt2_model

gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

In [5]:
class SimpleVQAModel(nn.Module):
    def __init__(self, clip_model, decoder_model, manual_cross_attn=None):
        super().__init__()
        self.clip = clip_model
        self.decoder = decoder_model
        self.manual_cross_attn = manual_cross_attn

        vision_hidden = clip_model.config.vision_config.hidden_size
        clip_text_hidden = clip_model.config.text_config.hidden_size
        decoder_hidden = decoder_model.config.n_embd

        self.image_proj = nn.Linear(vision_hidden, decoder_hidden)
        self.text_proj = nn.Linear(clip_text_hidden, decoder_hidden)
        # self.cross_attn = ManualCrossAttention(decoder_hidden)
    
    def forward(self, pixel_values, clip_input_ids, gpt_input_ids, labels=None):
        device = next(self.parameters()).device
        # 1. Extract CLIP features (frozen)
        with torch.no_grad():
            image_feats = self.clip.vision_model(pixel_values=pixel_values).last_hidden_state
            text_feats  = self.clip.text_model(input_ids=clip_input_ids).last_hidden_state

        # 2. Project to decoder hidden
        image_feats = self.image_proj(image_feats)
        text_feats  = self.text_proj(text_feats)

        # 3. Fuse (optional, can be identity)
        encoder_states = self.manual_cross_attn(text_feats, image_feats)     # [B, seq_ctx, d_model]

        # 4. Build attention masks
        encoder_attention_mask = torch.ones(encoder_states.size()[:2],dtype = torch.long, device=encoder_states.device)
        
        
        
        pad_id = getattr(gpt_tokenizer, "pad_token_id", None)
        if pad_id is None:
            # GPT-2 does not have a pad token → set pad = EOS
            gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
            pad_id = gpt_tokenizer.pad_token_id
            self.decoder.config.pad_token_id = pad_id
        gpt_input_ids = gpt_input_ids.to(device)
        decoder_attention_mask = (gpt_input_ids != gpt_tokenizer.pad_token_id).long()

        # 5. Call GPT-2 decoder **with cross-attention**
        outputs = self.decoder(
            input_ids=gpt_input_ids,
            attention_mask=decoder_attention_mask,
            encoder_hidden_states=encoder_states,
            encoder_attention_mask=encoder_attention_mask,
            labels=labels,
            return_dict=True
        )

        return outputs
        
    
    

In [6]:
class VQADataset(Dataset):
    def __init__(self, csv_path, image_folder, clip_processor, clip_tokenizer, gpt_tokenizer, max_length=64):
        self.df = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.clip_processor = clip_processor
        self.clip_tokenizer = clip_tokenizer
        self.gpt_tokenizer = gpt_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_folder, row["image_name"])

        image = Image.open(img_path).convert("RGB")
        pixel_values = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        question = str(row["question"])
        answer = str(row["answer"])

        # CLIP tokenization (for encoder side)
        clip_q = self.clip_tokenizer(
            question, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )

        # GPT-2: combined prompt "Question: X Answer: Y"
        prompt = f"Question: {question} Answer: {answer}"
        gpt_enc = self.gpt_tokenizer(
            prompt, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )
        gpt_input_ids = gpt_enc["input_ids"].squeeze(0)

        # Build labels: mask the question prefix so loss is only on the answer
        answer_prefix = f"Question: {question} Answer:"
        prefix_ids = self.gpt_tokenizer(answer_prefix, return_tensors="pt")["input_ids"].squeeze(0)
        prefix_len = len(prefix_ids)

        labels = gpt_input_ids.clone()
        labels[:prefix_len] = -100                                       # mask question portion
        labels[labels == self.gpt_tokenizer.pad_token_id] = -100         # mask padding

        # Safety: ensure at least one valid label token
        if (labels == -100).all():
            labels[prefix_len] = gpt_input_ids[prefix_len]

        return {
            "pixel_values": pixel_values,
            "clip_input_ids": clip_q["input_ids"].squeeze(0),
            "gpt_input_ids": gpt_input_ids,
            "labels": labels
        }


In [7]:

csv_path = r"E:/@IIT_BBS/@Sem 1/AI Lab/Project-GeoVQA/ENCODER/Dataset/vqa_filtered.csv"
image_folder = "E:/@IIT_BBS/@Sem 1/AI Lab/Project-GeoVQA/ENCODER/Dataset/images/train2014"

df = pd.read_csv(csv_path)
print(df["answer"].isna().sum(), "null answers")
print((df["answer"] == "").sum(), "empty answers")

0 null answers
0 empty answers


In [8]:
print("Loading Pretrained Models...")
clip_id = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_id)
clip_tokenizer = CLIPTokenizer.from_pretrained(clip_id)
clip_model = CLIPModel.from_pretrained(clip_id)

gpt_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
decoder_model = GPT2LMHeadModel.from_pretrained("gpt2")
print("Loading done!")

Loading Pretrained Models...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading done!


In [9]:
decoder = crossGPT2("gpt2")
manual_attn = ManualCrossAttention(decoder.config.n_embd)
vqa_model = SimpleVQAModel(clip_model, decoder, manual_cross_attn=manual_attn).to(device)


dataset = VQADataset(
    csv_path=csv_path,
    image_folder=image_folder,
    clip_processor = clip_processor,
    clip_tokenizer = clip_tokenizer,
    gpt_tokenizer= gpt_tokenizer
    )
    

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,         # this still helps with CPU→GPU transfer
)
# Freeze CLIP entirely
for p in vqa_model.clip.parameters():
    p.requires_grad = False

# Freeze GPT-2 except cross-attention related params
# Freeze CLIP entirely
for p in vqa_model.clip.parameters():
    p.requires_grad = False

# Unfreeze last 4 GPT-2 blocks fully + cross-attention in all blocks
for name, p in vqa_model.decoder.named_parameters():
    nl = name.lower()
    if "crossattention" in nl or "ln_cross_attn" in nl:
        p.requires_grad = True
    elif any(f"h.{i}." in name for i in range(8, 12)):  # unfreeze blocks 8-11
        p.requires_grad = True
    elif "ln_f" in nl:  # final layer norm
        p.requires_grad = True
    elif "lm_head" in nl:  # output head
        p.requires_grad = True
    else:
        p.requires_grad = False
# Ensure projection + manual fusion are trainable
for p in vqa_model.image_proj.parameters(): p.requires_grad = True
for p in vqa_model.text_proj.parameters(): p.requires_grad = True
if getattr(vqa_model, "manual_cross_attn", None) is not None:
    for p in vqa_model.manual_cross_attn.parameters(): p.requires_grad = True

# Build optimizer on trainable params
optimizer = torch.optim.AdamW([p for p in vqa_model.parameters() if p.requires_grad], lr=3e-5)
print("Trainable params:", sum(p.numel() for p in vqa_model.parameters() if p.requires_grad))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                                                 | Status     | 
----------------------------------------------------+------------+-
h.{0...11}.attn.bias                                | UNEXPECTED | 
transformer.h.{0...11}.ln_cross_attn.bias           | MISSING    | 
transformer.h.{0...11}.crossattention.c_attn.bias   | MISSING    | 
transformer.h.{0...11}.crossattention.q_attn.bias   | MISSING    | 
transformer.h.{0...11}.crossattention.c_proj.bias   | MISSING    | 
transformer.h.{0...11}.ln_cross_attn.weight         | MISSING    | 
transformer.h.{0...11}.crossattention.c_proj.weight | MISSING    | 
transformer.h.{0...11}.crossattention.q_attn.weight | MISSING    | 
transformer.h.{0...11}.crossattention.c_attn.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

Trainable params: 60068352


In [10]:
from torchinfo import summary
summary(vqa_model)

Layer (type:depth-idx)                                            Param #
SimpleVQAModel                                                    --
├─CLIPModel: 1-1                                                  1
│    └─CLIPTextTransformer: 2-1                                   --
│    │    └─CLIPTextEmbeddings: 3-1                               (25,336,320)
│    │    └─CLIPEncoder: 3-2                                      (37,828,608)
│    │    └─LayerNorm: 3-3                                        (1,024)
│    └─CLIPVisionTransformer: 2-2                                 --
│    │    └─CLIPVisionEmbeddings: 3-4                             (2,398,464)
│    │    └─LayerNorm: 3-5                                        (1,536)
│    │    └─CLIPEncoder: 3-6                                      (85,054,464)
│    │    └─LayerNorm: 3-7                                        (1,536)
│    └─Linear: 2-3                                                (393,216)
│    └─Linear: 2-4                    

In [12]:
# def train_model(epochs=3):
#     vqa_model.train()

#     for epoch in range(epochs):
#         total_loss = 0
#         pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

#         for batch in pbar:
#             pixel_values = batch["pixel_values"].to(device)
#             clip_input_ids = batch["clip_input_ids"].to(device)
#             gpt_input_ids = batch["gpt_input_ids"].to(device)
#             labels = batch["labels"].to(device)

#             outputs = vqa_model(pixel_values, clip_input_ids, gpt_input_ids, labels)
#             loss = outputs.loss

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             total_loss += loss.item()
#             pbar.set_postfix({"loss": f"{loss.item():.4f}"})

#     print("Training Complete!")

from tqdm.notebook import tqdm as tqdm_nb

def train_model(epochs=3):
    vqa_model.train()

    for epoch in range(epochs):
        total_loss = 0
        num_valid = 0
        pbar = tqdm_nb(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch in pbar:
            pixel_values = batch["pixel_values"].to(device)
            clip_input_ids = batch["clip_input_ids"].to(device)
            gpt_input_ids = batch["gpt_input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = vqa_model(pixel_values, clip_input_ids, gpt_input_ids, labels)
            loss = outputs.loss

            # Skip NaN batches instead of crashing
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"⚠️ Skipping batch with NaN/Inf loss")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in vqa_model.parameters() if p.requires_grad], max_norm=1.0
            )
            optimizer.step()

            total_loss += loss.item()
            num_valid += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / max(num_valid, 1)
        print(f"Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}")

    print("Training Complete!")

In [13]:
# batch = next(iter(train_loader))
# print("RAW batch:", batch)
# print("Type:", type(batch))

# if isinstance(batch, (list, tuple)):
#     for i, item in enumerate(batch):
#         print(f" Item {i}: type={type(item)}, value={repr(item)[:100]}")


In [14]:
batch = next(iter(train_loader))
print("BATCH TYPE:", type(batch))
if isinstance(batch, dict):
    print("keys:", list(batch.keys()))
    for k,v in batch.items():
        print(k, "->", type(v), getattr(v, "shape", str(type(v)) ) )
else:
    print("batch[0] type:", type(batch[0]))
    print("repr batch[0]:", repr(batch[0])[:300])


BATCH TYPE: <class 'dict'>
keys: ['pixel_values', 'clip_input_ids', 'gpt_input_ids', 'labels']
pixel_values -> <class 'torch.Tensor'> torch.Size([32, 3, 224, 224])
clip_input_ids -> <class 'torch.Tensor'> torch.Size([32, 64])
gpt_input_ids -> <class 'torch.Tensor'> torch.Size([32, 64])
labels -> <class 'torch.Tensor'> torch.Size([32, 64])


In [15]:

def warmup_vqa_from_dict(model, dataloader, steps=200, lr=1e-4, device=None):
    """
    Warmup training that accepts dataloaders returning a dict with keys:
    'pixel_values', 'clip_input_ids', 'gpt_input_ids', 'labels'
    """
    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    model.to(device)

    # Freeze CLIP entirely
    for p in model.clip.parameters():
        p.requires_grad = False

    # Freeze GPT-2 except cross-attention related params
    for name, p in model.decoder.named_parameters():
        nl = name.lower()
        if "crossattention" in nl or "ln_cross_attn" in nl or "cross_attn" in nl:
            p.requires_grad = True
        else:
            p.requires_grad = False

    # Ensure projection + manual fusion are trainable
    for p in model.image_proj.parameters(): p.requires_grad = True
    for p in model.text_proj.parameters(): p.requires_grad = True
    if getattr(model, "manual_cross_attn", None) is not None:
        for p in model.manual_cross_attn.parameters(): p.requires_grad = True

    # Build optimizer on trainable params
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

    model.train()
    data_iter = iter(dataloader)

    for step in range(steps):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        # --- Unpack dict produced by your loader ---
        if isinstance(batch, dict):
            pixel_values = batch["pixel_values"].to(device)         # [B, C, H, W]
            clip_input_ids = batch["clip_input_ids"].to(device)     # [B, L_clip]
            gpt_input_ids  = batch["gpt_input_ids"].to(device)      # [B, L_gpt]
            labels         = batch.get("labels")
            labels = labels.to(device) if labels is not None else None
        else:
            # fallback: try tuple unpack (older code paths)
            pixel_values, clip_input_ids, gpt_input_ids, labels = batch
            pixel_values = pixel_values.to(device)
            clip_input_ids = clip_input_ids.to(device)
            gpt_input_ids = gpt_input_ids.to(device)
            labels = labels.to(device) if labels is not None else None

        # forward + loss
        outputs = model(pixel_values, clip_input_ids, gpt_input_ids, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 20 == 0 or step == steps-1:
            print(f"[warmup {step+1}/{steps}] loss = {loss.item():.4f}")

    print("Warmup finished.")
    return model

warmup_vqa_from_dict(
    model=vqa_model,
    dataloader=train_loader,
    steps=150,   # 50-200 usually enough to warm cross-attn
    lr=1e-4,
    device=device
)


Trainable params: 31715328


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[warmup 1/150] loss = 8.5618
[warmup 21/150] loss = 5.9783
[warmup 41/150] loss = 5.4079
[warmup 61/150] loss = 3.9902
[warmup 81/150] loss = 3.8229
[warmup 101/150] loss = 2.9445
[warmup 121/150] loss = 2.3055
[warmup 141/150] loss = 2.9660
[warmup 150/150] loss = 1.4855
Warmup finished.


SimpleVQAModel(
  (clip): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bias=True)
 

In [16]:
train_model(epochs = 3)

Epoch 1/3:   0%|          | 0/13868 [00:00<?, ?it/s]

Epoch 1 — Avg Loss: 2.3473


Epoch 2/3:   0%|          | 0/13868 [00:00<?, ?it/s]

Epoch 2 — Avg Loss: 2.0196


Epoch 3/3:   0%|          | 0/13868 [00:00<?, ?it/s]

Epoch 3 — Avg Loss: 1.8537
Training Complete!


In [17]:
from datetime import datetime
import torch

print("Saving Model...")

# Use a filename-safe timestamp
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
print(f"Time: {timestamp}")

torch.save(vqa_model.state_dict(), f"saved-models/vqa_model_{timestamp}.pth")
torch.save(optimizer.state_dict(), f"saved-models/vqa_optimizer_{timestamp}.pth")


Saving Model...
Time: 2026-02-16_07-48-19


In [3]:
vqa_model.load_state_dict(torch.load(f"saved-models/vqa_model_{timestamp}.pth"))
vqa_model.to(device)

NameError: name 'vqa_model' is not defined

In [11]:
vqa_model.load_state_dict(torch.load(r"saved-models\vqa_model_2026-02-16_07-48-19.pth"))
vqa_model.to(device)

SimpleVQAModel(
  (clip): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bias=True)
 

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

#---- image path ----
img_path = r"E:/@IIT_BBS/@Sem 1/AI Lab/Project-GeoVQA/ENCODER/Dataset/images/test2014/COCO_test2014_000000000456.jpg"

# --- Helper: load image into widget bytes ---
def load_image_widget_bytes(path, max_size=(300, 300)):
    """Open an image, resize for preview, return PNG bytes."""
    img = Image.open(path).convert('RGB')
    img.thumbnail(max_size)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return buf.getvalue()

# --- UI Elements ---
img_path_input = widgets.Text(
    value=img_path,
    placeholder='Enter image path...',
    description='Image:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '60px'}
)

# Image preview widget — shows the actual image instead of just the path
img_preview = widgets.Image(
    format='png',
    width=300,
    height=300,
    layout=widgets.Layout(border='1px solid #ccc', margin='5px 0')
)
preview_status = widgets.HTML(value='')

def update_preview(path):
    """Update the image preview widget from a file path."""
    try:
        img_preview.value = load_image_widget_bytes(path)
        preview_status.value = ''
    except Exception as e:
        img_preview.value = b''
        preview_status.value = f'<span style="color:red">⚠️ {e}</span>'

# Load initial preview
update_preview(img_path)

# Update preview when path changes
def on_path_change(change):
    update_preview(change['new'].strip())
img_path_input.observe(on_path_change, names='value')

question_input = widgets.Text(
    value='What is shown in the image?',
    placeholder='Ask a question...',
    description='Question:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': '60px'}
)

ask_btn = widgets.Button(
    description='Ask',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='120px')
)

output_area = widgets.Output()

def on_ask(btn):
    with output_area:
        clear_output(wait=True)
        path = img_path_input.value.strip()
        question = question_input.value.strip()
        if not path or not question:
            print("⚠️ Please provide both an image path and a question.")
            return

        try:
            image = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"❌ Could not open image: {e}")
            return

        # Update preview widget with the queried image
        update_preview(path)

        # --- Run inference ---
        vqa_model.eval()
        pixel_values = clip_processor(images=image, return_tensors="pt")["pixel_values"].to(device)
        clip_ids = clip_tokenizer(question, return_tensors="pt")["input_ids"].to(device)

        prompt = f"Question: {question} Answer:"
        gpt_ids = gpt_tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

        with torch.no_grad():
            img_feats = vqa_model.clip.vision_model(pixel_values=pixel_values).last_hidden_state
            txt_feats = vqa_model.clip.text_model(input_ids=clip_ids).last_hidden_state
            img_feats = vqa_model.image_proj(img_feats)
            txt_feats = vqa_model.text_proj(txt_feats)
            encoder_states = vqa_model.manual_cross_attn(txt_feats, img_feats)
            encoder_attention_mask = torch.ones(
                encoder_states.size()[:2], dtype=torch.long, device=device
            )

            generated_ids = gpt_ids.clone()
            answer_tokens = []
            for _ in range(20):
                outputs = vqa_model.decoder(
                    input_ids=generated_ids,
                    encoder_hidden_states=encoder_states,
                    encoder_attention_mask=encoder_attention_mask,
                    return_dict=True
                )
                next_token_logits = outputs.logits[:, -1, :]
                next_token_id = torch.argmax(next_token_logits, dim=-1)

                if next_token_id.item() == gpt_tokenizer.eos_token_id:
                    break

                word = gpt_tokenizer.decode([next_token_id.item()])
                answer_tokens.append(word)

                if '.' in word or '\n' in word:
                    break

                generated_ids = torch.cat(
                    [generated_ids, next_token_id.unsqueeze(0)], dim=1
                )

        print(f"\n🔹 Question: {question}")
        print(f"🔸 Answer: {''.join(answer_tokens).strip()}")

ask_btn.on_click(on_ask)

# Submit on Enter in the question box
def on_enter(change):
    on_ask(None)
question_input.on_submit(on_enter)

# --- Layout ---
ui = widgets.VBox([
    widgets.HTML("<h3>🖼️ VQA Interactive Demo</h3>"),
    img_path_input,
    preview_status,
    img_preview,
    question_input,
    ask_btn,
    output_area
], layout=widgets.Layout(padding='10px'))

display(ui)

C:\Users\Sagnik\AppData\Local\Temp\ipykernel_25300\693610293.py:138: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  question_input.on_submit(on_enter)
